# 🐛 CoBug AI Learning Platform — Colab Runner

**Author:** ARJUN V B  
**Project:** AI-Based Student Behaviour Analysis and Personalised Learning Pathways  
**GitHub:** [Arjun-115/CoBug-ai-learning-platform](https://github.com/Arjun-115/CoBug-ai-learning-platform)

---

This notebook launches the CoBug AI Learning Platform inside Google Colab and gives you a live public URL to use it — no local setup needed.

---

## ✅ Before you start — two free keys needed

| Key | Where to get it | Time |
|-----|----------------|------|
| **Groq API Key** | [console.groq.com](https://console.groq.com) → API Keys → Create | ~1 min |
| **ngrok Auth Token** | [dashboard.ngrok.com](https://dashboard.ngrok.com) → Your Authtoken | ~1 min |

---

## 📦 Getting the project files into Colab

**Option A — Clone from GitHub (recommended):**
The project is already on GitHub. In Cell 1, set `UPLOAD_METHOD = 'github'` — no file upload needed.

**Option B — Upload ZIP:**
If you have `ai-learning-platform.zip`, click the 📁 folder icon on the left, upload it, then set `UPLOAD_METHOD = 'zip'` in Cell 1.

---

## ▶ Run order

**Cell 1** → Environment setup (install Node.js + dependencies)  
**Cell 2** → Paste your API keys  
**Cell 3** → Build and launch — gives you a live URL  
**Cell 4** → Export learning data as CSV (run anytime after using the platform)  
**Cell 5** → Stop all services cleanly  

> **Note:** Cells 1–3 take about 3–4 minutes total on first run (mostly npm install). After that, re-running Cell 3 is fast.

---
## Cell 1 — Environment Setup
*Fetches the project, installs Node.js, and sets up all dependencies. Run once per Colab session.*

In [ ]:
import subprocess, sys, os, zipfile, shutil, tempfile

# ============================================================
# CHOOSE HOW TO GET THE PROJECT FILES
# 'github' — clone directly from GitHub (no upload needed)
# 'zip'    — upload ai-learning-platform.zip via the file panel
# ============================================================
UPLOAD_METHOD = 'github'
GITHUB_REPO   = 'https://github.com/Arjun-115/CoBug-ai-learning-platform.git'
# ============================================================

platform_dir = '/content/ai-learning-platform'
zip_path     = '/content/ai-learning-platform.zip'

# ── Step 1: Get project files ────────────────────────────────
if os.path.exists(platform_dir):
    print('[1/5] Project folder already exists, skipping download.')
else:
    if UPLOAD_METHOD == 'github':
        print(f'[1/5] Cloning from GitHub...')
        r = subprocess.run(
            ['git', 'clone', GITHUB_REPO, platform_dir],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            raise RuntimeError(
                f'Git clone failed:\n{r.stderr}\n'
                'Make sure the repo is public and the URL is correct.'
            )
        print('  Cloned successfully.')

    elif UPLOAD_METHOD == 'zip':
        print('[1/5] Extracting zip...')
        if not os.path.exists(zip_path):
            raise FileNotFoundError(
                'ai-learning-platform.zip not found.\n'
                'Upload it using the 📁 folder icon on the left, then re-run this cell.'
            )
        tmp = tempfile.mkdtemp()
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(tmp)
        entries        = os.listdir(tmp)
        subdirs        = [e for e in entries if os.path.isdir(os.path.join(tmp, e))]
        has_root_files = any(os.path.isfile(os.path.join(tmp, e)) for e in entries)
        if subdirs and not has_root_files:
            shutil.move(os.path.join(tmp, subdirs[0]), platform_dir)
        else:
            shutil.move(tmp, platform_dir)
        print('  Extracted successfully.')
    else:
        raise ValueError('UPLOAD_METHOD must be "github" or "zip".')

# Check the folder has what we expect
for folder in ['backend', 'frontend']:
    if not os.path.isdir(os.path.join(platform_dir, folder)):
        raise FileNotFoundError(
            f'The "{folder}" folder is missing from the project.\n'
            f'Re-check the zip or repo structure.'
        )
print('  Project structure OK.')

# ── Step 2: Node.js ──────────────────────────────────────────
print('[2/5] Checking Node.js...')
try:
    node_ver = subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    node_ver = ''

if not any(node_ver.startswith(v) for v in ['v18', 'v20', 'v22']):
    print('  Node.js not found or too old — installing v20 (takes ~1 min)...')
    r = subprocess.run(
        'curl -fsSL https://deb.nodesource.com/setup_20.x | bash -',
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'Node.js setup script failed:\n{r.stderr[-500:]}')
    r = subprocess.run(
        'apt-get install -y nodejs',
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'Node.js install failed:\n{r.stderr[-500:]}')
    node_ver = subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip()
print(f'  Node.js {node_ver} ready.')

# ── Step 3: pyngrok ──────────────────────────────────────────
print('[3/5] Installing pyngrok...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'pyngrok', '-q'],
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f'pyngrok install failed:\n{r.stderr[-400:]}')
print('  pyngrok ready.')

# ── Step 4: Backend packages ──────────────────────────────────
print('[4/5] Installing backend packages (this may take a minute)...')
r = subprocess.run(
    ['npm', 'install'],
    cwd=f'{platform_dir}/backend',
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f'Backend npm install failed:\n{r.stderr[-800:]}')
print('  Backend packages ready.')

# ── Step 5: Frontend packages ─────────────────────────────────
print('[5/5] Installing frontend packages...')
r = subprocess.run(
    ['npm', 'install'],
    cwd=f'{platform_dir}/frontend',
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f'Frontend npm install failed:\n{r.stderr[-800:]}')
print('  Frontend packages ready.')

print()
print('✅ Setup complete! Proceed to Cell 2.')

---
## Cell 2 — API Keys
*Paste your Groq and ngrok keys below, then run this cell.*

In [ ]:
import os

# ============================================================
#  PASTE YOUR KEYS HERE — replace the placeholder strings
# ============================================================
GROQ_API_KEY     = 'gsk_your_groq_key_here'   # console.groq.com → API Keys
NGROK_AUTH_TOKEN = 'your_ngrok_token_here'     # dashboard.ngrok.com → Your Authtoken
GITHUB_TOKEN     = ''                          # optional — raises GitHub rate limit
# ============================================================

if not GROQ_API_KEY.strip() or GROQ_API_KEY.strip() == 'gsk_your_groq_key_here':
    raise ValueError(
        'GROQ_API_KEY is missing.\n'
        'Sign up at https://console.groq.com, go to API Keys, create a key, and paste it above.'
    )

if not NGROK_AUTH_TOKEN.strip() or NGROK_AUTH_TOKEN.strip() == 'your_ngrok_token_here':
    raise ValueError(
        'NGROK_AUTH_TOKEN is missing.\n'
        'Sign up at https://dashboard.ngrok.com, copy your authtoken, and paste it above.'
    )

# Write the backend .env file
backend_dir = '/content/ai-learning-platform/backend'
os.makedirs(backend_dir, exist_ok=True)
env_path = os.path.join(backend_dir, '.env')
with open(env_path, 'w') as f:
    f.write(f'GROQ_API_KEY={GROQ_API_KEY.strip()}\n')
    f.write(f'GITHUB_TOKEN={GITHUB_TOKEN.strip()}\n')
    f.write('PORT=5000\n')

# Store ngrok token in memory only — not written to disk for security
os.environ['COBUG_NGROK_TOKEN'] = NGROK_AUTH_TOKEN.strip()

print('✅ Keys saved successfully.')
print('   Proceed to Cell 3 to build and launch the platform.')

---
## Cell 3 — Build and Launch
*Compiles the frontend, starts the backend server, and opens a public URL via ngrok.*  
*Keep this cell running the whole time you use the platform.*

In [ ]:
import subprocess, sys, os, shutil, time, urllib.request, threading

platform_dir = '/content/ai-learning-platform'
frontend_dir = f'{platform_dir}/frontend'
backend_dir  = f'{platform_dir}/backend'
GITHUB_REPO  = 'https://github.com/Arjun-115/CoBug-ai-learning-platform.git'

# ── Auto-recover: re-clone if the project folder is missing ──
# This handles the case where the runtime restarted after Cell 1 ran
if not os.path.isdir(platform_dir):
    print('Project folder not found — cloning from GitHub...')
    r = subprocess.run(['git', 'clone', GITHUB_REPO, platform_dir], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed. Check your internet connection.\n{r.stderr}')
    print('  Cloned.')

# ── Auto-recover: re-install packages if node_modules is missing ──
if not os.path.isdir(f'{backend_dir}/node_modules'):
    print('Backend packages missing — running npm install...')
    subprocess.run(['npm', 'install'], cwd=backend_dir, capture_output=True)
    print('  Done.')

if not os.path.isdir(f'{frontend_dir}/node_modules'):
    print('Frontend packages missing — running npm install...')
    subprocess.run(['npm', 'install'], cwd=frontend_dir, capture_output=True)
    print('  Done.')

# ── Install pyngrok if missing ─────────────────────────────────
try:
    from pyngrok import ngrok, conf
except ImportError:
    print('Installing pyngrok...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok', '-q'], capture_output=True)
    from pyngrok import ngrok, conf

# ── Pre-flight checks ──────────────────────────────────────────
if not os.path.isdir(frontend_dir):
    raise RuntimeError('Frontend folder still missing after clone. Check the GitHub repo is public.')
if not os.path.exists(f'{backend_dir}/.env'):
    raise RuntimeError('API keys not found — run Cell 2 first and paste your Groq + ngrok keys.')
if not os.environ.get('COBUG_NGROK_TOKEN'):
    raise RuntimeError('ngrok token missing from memory — run Cell 2 again (runtime may have restarted).')

# ── 1. Build frontend ─────────────────────────────────────────
print('[1/4] Building frontend (takes ~60 seconds)...')
build_env = os.environ.copy()
build_env['NODE_OPTIONS'] = '--max-old-space-size=512'

r = subprocess.run(
    ['npm', 'run', 'build'],
    cwd=frontend_dir,
    env=build_env,
    capture_output=True,
    text=True
)

# Vite writes deprecation warnings to stderr but still exits 0 — only fail on real errors
dist_path = os.path.join(frontend_dir, 'dist', 'index.html')
if r.returncode != 0 or not os.path.exists(dist_path):
    print('❌ Build failed. Output:')
    print(r.stdout[-1200:])
    print(r.stderr[-1200:])
    raise RuntimeError('Frontend build failed. See output above.')
print('  Frontend built.')

# ── 2. Copy dist → backend/public ────────────────────────────
print('[2/4] Copying static files to backend...')
public_dest = f'{backend_dir}/public'
if os.path.exists(public_dest):
    shutil.rmtree(public_dest)
shutil.copytree(f'{frontend_dir}/dist', public_dest)
print('  Static files ready.')

# ── 3. Start backend server ───────────────────────────────────
print('[3/4] Starting backend server...')
subprocess.run(['pkill', '-f', 'node'], capture_output=True)
time.sleep(1.5)

log_path = '/content/server.log'
log_file = open(log_path, 'w')
server = subprocess.Popen(
    ['node', 'server.js'],
    cwd=backend_dir,
    stdout=log_file,
    stderr=log_file
)

print('  Waiting for server', end='', flush=True)
is_up = False
for _ in range(40):
    time.sleep(1)
    print('.', end='', flush=True)
    if server.poll() is not None:  # process exited unexpectedly
        log_file.flush()
        print('\nServer crashed on startup. Log:')
        with open(log_path) as lf: print(lf.read())
        raise RuntimeError('Server crashed — check the log above.')
    try:
        with urllib.request.urlopen('http://localhost:5000/api/health', timeout=2) as resp:
            if resp.status == 200:
                print(' UP!')
                is_up = True
                break
    except Exception:
        pass

if not is_up:
    log_file.flush()
    print('\nServer did not respond in 40 seconds. Log:')
    with open(log_path) as lf: print(lf.read())
    raise RuntimeError('Server timed out — check the log above.')

# ── 4. Open ngrok tunnel ──────────────────────────────────────
print('[4/4] Opening public tunnel...')

conf.get_default().auth_token = os.environ['COBUG_NGROK_TOKEN']
try:
    ngrok.kill()
    time.sleep(0.8)
except Exception:
    pass

tunnel     = ngrok.connect(5000)
public_url = tunnel.public_url

print()
print('=' * 62)
print('  ✅  CoBug AI Learning Platform is LIVE')
print(f'  🌐  {public_url}')
print('=' * 62)
print()
print('Open the URL above in your browser.')
print('This cell must stay running — stopping it shuts the server down.')
print()

# Stream server logs live so you can see what's happening
def _tail_log():
    with open(log_path, 'r') as f:
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(f'  [server] {line.rstrip()}')
            else:
                time.sleep(0.3)

threading.Thread(target=_tail_log, daemon=True).start()

# Keep cell alive; auto-restart server if it crashes
try:
    while True:
        time.sleep(10)
        if server.poll() is not None:
            print('  [server] Crashed — restarting...')
            log_file2 = open(log_path, 'a')
            server = subprocess.Popen(
                ['node', 'server.js'],
                cwd=backend_dir,
                stdout=log_file2,
                stderr=log_file2
            )
            print(f'  [server] Restarted (PID {server.pid})')
except KeyboardInterrupt:
    print('\nStopped by user. Run Cell 5 to clean up.')

---
## Cell 4 — Export Learning Data to CSV
*Exports all recorded student behaviour data to a CSV and downloads it to your computer.*  
*Run this any time after completing exercises on the platform.*

In [ ]:
import sqlite3, csv, os

db_path  = '/content/ai-learning-platform/backend/learning.db'
csv_path = '/content/cobug_dataset.csv'

if not os.path.exists(db_path):
    print('No database file found.')
    print('Complete at least one quiz or exercise on the platform first, then run this cell again.')
else:
    conn   = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute('''
            SELECT id, user_id, topic, language, skill_level,
                   score, total, time_spent, hints_used,
                   behavior_tag, recommended_action, created_at
            FROM scores
            ORDER BY created_at ASC
        ''')
        rows    = cursor.fetchall()
        columns = [d[0] for d in cursor.description]

        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(columns)
            writer.writerows(rows)

        print(f'✅ Exported {len(rows)} record(s) to {csv_path}')

        try:
            from google.colab import files
            files.download(csv_path)
            print('   Download started in your browser.')
        except Exception:
            print(f'   To download: click the 📁 folder icon on the left and find cobug_dataset.csv')

    except sqlite3.OperationalError as e:
        print(f'Database error: {e}')
    finally:
        conn.close()

---
## Cell 5 — Stop All Services
*Shuts down the backend server and closes the ngrok tunnel cleanly.*

In [ ]:
import subprocess

print('Closing ngrok tunnel...')
try:
    from pyngrok import ngrok
    ngrok.kill()
    print('  Done.')
except Exception as e:
    print(f'  Already stopped ({e})')

print('Stopping Node.js server...')
subprocess.run(['pkill', '-f', 'node'], capture_output=True)
print('  Done.')
print()
print('All services stopped.')

---
## Acknowledgements

I would like to thank everyone who supported the development of this project.

First, I am grateful to my project supervisor and the faculty at my institution for their guidance, feedback, and encouragement throughout the research and development process.

I would like to acknowledge the following tools and platforms that made this project possible:

- **[Groq](https://groq.com)** — for providing fast, free access to the LLaMA 3.3 language model through their API, which powers all AI features in this platform.
- **[ngrok](https://ngrok.com)** — for enabling public URL tunnelling from Google Colab, making it straightforward to demo and share the running application.
- **[Google Colab](https://colab.research.google.com)** — for providing a free cloud environment that allows this project to run without any local installation.
- **[React](https://react.dev)**, **[Vite](https://vitejs.dev)**, and **[Tailwind CSS](https://tailwindcss.com)** — for the frontend framework and tooling.
- **[Express.js](https://expressjs.com)** and **[sql.js](https://sql.js.org)** — for the backend API and in-browser SQLite database.
- **[GitHub](https://github.com)** — for version control and project hosting.

Finally, I would like to thank the open-source community whose libraries, documentation, and shared knowledge made it possible to build a working AI-powered learning platform as an individual project.

---
*CoBug AI Learning Platform — ARJUN V B*